# Part 1: Data Acquisition, Cleaning, and Exploratory Analysis
This notebook loads the raw data, carries out basic cleaning steps (missing value treatment, duplicate removal, type conversion), displays descriptive statistics, and saves diagnostic plots.

### Load and Inspect Raw Data

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Loading the raw data file...")
dataframe = pd.read_csv("raw_data.csv" if os.path.exists("raw_data.csv") else "part1/raw_data.csv")
print("Rows: " + str(dataframe.shape[0]) + " | Columns: " + str(dataframe.shape[1]))
dataframe.head()

In [ ]:
print("Columns and types:")
print(dataframe.dtypes)
original_memory = dataframe.memory_usage(deep=True).sum()

### Null Value Analysis

In [ ]:
print("--- Analysing Missing Values ---")
total_rows = len(dataframe)
for col in dataframe.columns:
    nulls = dataframe[col].isnull().sum()
    pct = (nulls / total_rows) * 100
    print("Column: " + col + " | Missing: " + str(nulls) + " (" + str(round(pct, 2)) + "%)")

### Duplicate Detection and Removal

In [ ]:
print("--- Checking for Duplicates ---")
duplicate_count = dataframe.duplicated().sum()
print("Found " + str(duplicate_count) + " duplicate rows.")
dataframe = dataframe.drop_duplicates()
print("Shape after removing duplicates: " + str(dataframe.shape))

### Data Type Correction

In [ ]:
print("--- Fixing Incorrect Data Types ---")
empty_space_rows = dataframe[dataframe['TotalCharges'].str.strip() == ""]
print("Blank spaces in TotalCharges: " + str(len(empty_space_rows)))

dataframe['TotalCharges'] = pd.to_numeric(dataframe['TotalCharges'], errors='coerce')
print("TotalCharges dtype: " + str(dataframe['TotalCharges'].dtype))
print("Missing TotalCharges values: " + str(dataframe['TotalCharges'].isnull().sum()))

In [ ]:
# Convert repetitive string columns to category type
columns_to_category = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod', 'Churn'
]
for col in columns_to_category:
    dataframe[col] = dataframe[col].astype('category')

new_memory = dataframe.memory_usage(deep=True).sum()
saved_mem = original_memory - new_memory
saved_pct = (saved_mem / original_memory) * 100
print("Memory saved: " + str(saved_mem) + " bytes (" + str(round(saved_pct, 2)) + "%)")

### Descriptive Statistics and Skewness

In [ ]:
print(dataframe[['tenure', 'MonthlyCharges', 'TotalCharges']].describe())
print("tenure skewness: " + str(round(dataframe['tenure'].skew(), 4)))
print("MonthlyCharges skewness: " + str(round(dataframe['MonthlyCharges'].skew(), 4)))
print("TotalCharges skewness: " + str(round(dataframe['TotalCharges'].skew(), 4)))

### Outlier Detection with IQR

In [ ]:
numeric_columns = ['MonthlyCharges', 'TotalCharges']
for col in numeric_columns:
    q1 = dataframe[col].quantile(0.25)
    q3 = dataframe[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = dataframe[(dataframe[col] < lower_bound) | (dataframe[col] > upper_bound)]
    print("Column: " + col)
    print("  Bounds: [" + str(round(lower_bound, 2)) + ", " + str(round(upper_bound, 2)) + "] | Outliers: " + str(len(outliers)))

### Visualizations

In [ ]:
# 1. Line plot of MonthlyCharges
plt.figure(figsize=(8, 4))
plt.plot(dataframe.index, dataframe['MonthlyCharges'], color='blue', alpha=0.5)
plt.title("Monthly Charges Line Plot")
plt.xlabel("Index")
plt.ylabel("Monthly Charges")
plt.savefig("line_plot_monthly_charges.png" if os.path.exists("cleaned_data.csv") else "part1/line_plot_monthly_charges.png")
plt.show()

In [ ]:
# 2. Bar chart comparing mean charges by contract
plt.figure(figsize=(6, 4))
mean_charges = dataframe.groupby('Contract', observed=True)['MonthlyCharges'].mean()
plt.bar(mean_charges.index, mean_charges.values, color='orange', edgecolor='black')
plt.title("Average Monthly Charges by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Average Monthly Charges")
plt.savefig("bar_chart_monthly_charges_by_contract.png" if os.path.exists("cleaned_data.csv") else "part1/bar_chart_monthly_charges_by_contract.png")
plt.show()

In [ ]:
# 3. Histogram of most skewed column (TotalCharges)
plt.figure(figsize=(6, 4))
plt.hist(dataframe['TotalCharges'].dropna(), bins=20, color='green', edgecolor='black', alpha=0.7)
plt.title("Histogram of Total Charges")
plt.xlabel("Total Charges")
plt.ylabel("Frequency")
plt.savefig("histogram_most_skewed.png" if os.path.exists("cleaned_data.csv") else "part1/histogram_most_skewed.png")
plt.show()

In [ ]:
# 4. Scatter plot between tenure and TotalCharges
plt.figure(figsize=(6, 4))
plt.scatter(dataframe['tenure'], dataframe['TotalCharges'], color='purple', alpha=0.5)
plt.title("Tenure vs Total Charges")
plt.xlabel("Tenure (months)")
plt.ylabel("Total Charges")
plt.savefig("scatter_plot_tenure_vs_total_charges.png" if os.path.exists("cleaned_data.csv") else "part1/scatter_plot_tenure_vs_total_charges.png")
plt.show()

In [ ]:
# 5. Box plot of MonthlyCharges split by Churn
plt.figure(figsize=(6, 4))
sns.boxplot(x='Churn', y='MonthlyCharges', data=dataframe, hue='Churn', palette=['green', 'red'], legend=False)
plt.title("Monthly Charges by Churn Status")
plt.xlabel("Churned?")
plt.ylabel("Monthly Charges")
plt.savefig("box_plot_monthly_charges_by_churn.png" if os.path.exists("cleaned_data.csv") else "part1/box_plot_monthly_charges_by_churn.png")
plt.show()

### Correlation Heatmap

In [ ]:
numeric_df = dataframe[['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']]
pearson_corr = numeric_df.corr()
print("Pearson Correlation:")
print(pearson_corr)

plt.figure(figsize=(6, 5))
sns.heatmap(pearson_corr, annot=True, cmap='Blues')
plt.title("Pearson Correlation Heatmap")
plt.savefig("correlation_heatmap.png" if os.path.exists("cleaned_data.csv") else "part1/correlation_heatmap.png")
plt.show()

### Imputation Strategy Comparison

In [ ]:
tc_mean = dataframe['TotalCharges'].mean()
tc_median = dataframe['TotalCharges'].median()
print("TotalCharges - Mean: " + str(round(tc_mean, 4)) + " | Median: " + str(round(tc_median, 4)))

# Impute missing values with the median
dataframe['TotalCharges'] = dataframe['TotalCharges'].fillna(tc_median)
print("Missing values in TotalCharges after filling: " + str(dataframe['TotalCharges'].isnull().sum()))

### Spearman Rank Correlation

In [ ]:
spearman_corr = numeric_df.corr(method='spearman')
print("Spearman Correlation:")
print(spearman_corr)

diff_corr = (spearman_corr - pearson_corr).abs()
print("\nAbsolute Difference Matrix (|Spearman - Pearson|):")
print(diff_corr)

### Grouped Aggregation & Export

In [ ]:
grouped = dataframe.groupby('Contract', observed=True)['MonthlyCharges'].agg(['mean', 'std', 'count'])
print(grouped)

# Save cleaned dataset
dataframe.to_csv("cleaned_data.csv" if os.path.exists("cleaned_data.csv") else "part1/cleaned_data.csv", index=False)
print("Cleaned dataset saved successfully!")